In [ ]:
import pandas as pd
from pathlib import Path
import os
import json
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# utils

def normalize_text(s: str) -> str:
    """Lowercase and strip punctuation."""
    s = s.lower()
    s = re.sub(r'[^\w\s]', '', s)
    return s.strip()


def text_match(s1: str, s2: str) -> bool:
    """Exact match after normalization."""
    return normalize_text(str(s1)) == normalize_text(str(s2))


def cosine_sim(s1: str, s2: str) -> float:
    """Cosine similarity between two strings using TF-IDF character n-grams."""
    if not s1 and not s2:
        return 1.0
    if not s1 or not s2:
        return 0.0
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3))
    try:
        tfidf = vec.fit_transform([normalize_text(s1), normalize_text(s2)])
        return float(cosine_similarity(tfidf[0], tfidf[1])[0][0])
    except ValueError:
        return 0.0


# ── row matching ───────────────────────────────────────────────────────────────

def find_matching_row(source_row: pd.Series, target_df: pd.DataFrame,
                      match_threshold: float = 0.6) -> pd.Series | None:
    """
    Find the best-matching row in `target_df` for `source_row` using:
      - relation_name    : cosine similarity → 0..1
      - relation_address : cosine similarity → 0..1

    Returns the best candidate above `match_threshold`, or None.
    Used symmetrically for both pred→GT and GT→pred lookups.
    """
    best_row = None
    best_score = -1.0

    for _, row in target_df.iterrows():
        name_score    = cosine_sim(source_row['relation_name'],    row['relation_name'])
        # address_score = cosine_sim(source_row['relation_address'], row['relation_address'])
        total = name_score # + address_score

        if total > best_score:
            best_score = total
            best_row   = row

    return best_row if best_score >= match_threshold else None


# ── field-level comparison ─────────────────────────────────────────────────────

def compare_row(pred_row: pd.Series, gt_row: pd.Series,
                address_threshold: float = 0.6,
                name_threshold: float = 0.6,
                company_threshold: float = 0.6) -> dict:
    """
    Compare one prediction row against its matched ground-truth row field by field.

    Returns a dict with cosine scores, boolean match flags, and overall_match.
    """
    exact_fields = ['relation_country', 'relation_nationality',
                    'relation_type', 'relationship']
    result = {}

    company_score = cosine_sim(pred_row['company_name'], gt_row['company_name'])
    result['company_name_cosine'] = round(company_score, 4)
    result['company_name_match']  = company_score >= company_threshold

    name_score = cosine_sim(pred_row['relation_name'], gt_row['relation_name'])
    result['relation_name_cosine'] = round(name_score, 4)
    result['relation_name_match']  = name_score >= name_threshold

    addr_score = cosine_sim(pred_row['relation_address'], gt_row['relation_address'])
    result['relation_address_cosine'] = round(addr_score, 4)
    result['relation_address_match']  = addr_score >= address_threshold

    for field in exact_fields:
        result[f'{field}_match'] = text_match(pred_row[field], gt_row[field])

    result['overall_match'] = all(v for k, v in result.items() if k.endswith('_match'))
    return result


# ── full evaluation ────────────────────────────────────────────────────────────

_COSINE_FIELDS = ['company_name', 'relation_name', 'relation_address']
_EXACT_FIELDS  = ['relation_country', 'relation_nationality', 'relation_type', 'relationship']
_ALL_FIELDS    = _COSINE_FIELDS + _EXACT_FIELDS

def _empty_comparison(overall: bool = False) -> dict:
    """Return a zeroed-out comparison dict (used for FP / FN rows)."""
    d = {f'{f}_match': False for f in _ALL_FIELDS}
    d.update({f'{f}_cosine': 0.0 for f in _COSINE_FIELDS})
    d['overall_match'] = overall
    return d


def evaluate_predictions(model_predictions: pd.DataFrame,
                         ground_truth: pd.DataFrame,
                         address_threshold: float = 0.6,
                         name_threshold: float = 0.6,
                         company_threshold: float = 0.6
                         ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    """
    Full precision/recall evaluation.

    Returns
    -------
    tp_df   : matched pairs with field-level comparison (true positives)
    fp_df   : predictions with no GT match              (false positives)
    fn_df   : GT rows never predicted                   (false negatives)
    metrics : dict with precision, recall, F1
    """
    kw = dict(address_threshold=address_threshold,
              name_threshold=name_threshold,
              company_threshold=company_threshold)

    tp_records, fp_records = [], []
    matched_gt_indices = set()

    # ── loop over predictions → find TP and FP ────────────────────────────────
    for _, pred_row in model_predictions.iterrows():
        gt_row = find_matching_row(pred_row, ground_truth)

        base = {f'pred_{k}': v for k, v in pred_row.items()}

        if gt_row is not None:
            matched_gt_indices.add(gt_row.name)
            gt_base = {f'gt_{k}': v for k, v in gt_row.items()}
            comparison = compare_row(pred_row, gt_row, **kw)
            tp_records.append({**base, **gt_base, **comparison})
        else:
            fp_records.append({**base, **_empty_comparison()})

    # ── loop over GT → find FN (GT rows never matched) ────────────────────────
    fn_records = []
    for idx, gt_row in ground_truth.iterrows():
        if idx not in matched_gt_indices:
            fn_records.append({f'gt_{k}': v for k, v in gt_row.items()})

    tp_df = pd.DataFrame(tp_records)
    fp_df = pd.DataFrame(fp_records)
    fn_df = pd.DataFrame(fn_records)

    return tp_df, fp_df, fn_df

def calc_metrics(fp_df, tp_df, fn_df):
    tp = len(tp_df[tp_df['overall_match']]) if not tp_df.empty else 0
    fp = len(fp_df) + (len(tp_df) - tp)   # unmatched preds + matched-but-wrong fields
    fn = len(fn_df)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)

    metrics = dict(TP=tp, FP=fp, FN=fn,
                   precision=round(precision, 4),
                   recall=round(recall, 4),
                   f1=round(f1, 4))
    return metrics

In [ ]:
# load ground truth
# ground_truth_folder = Path("/home/ubuntu/data/template_extraction_0/json_output_ground_truth")

csv_path = Path("/home/ubuntu/data/template_extraction_0/Template_KYC_Manual - Annotation.csv")
gt_csv = pd.read_csv(csv_path)
gt_csv.fillna("", inplace=True)

print("Ground truth finds %d relations" % gt_csv.shape[0])
gt_csv.head()

In [ ]:
# load model predictions
predictions_file = Path("/home/mfridman/olsomet-table-extract/output_qianfan.csv")
p = pd.read_csv(predictions_file)

p.fillna("", inplace=True)
p.head()

In [ ]:
# EDA
# search for filename "<document-id>_1"
fn = "<document-id>.pdf"
pg = 1
p[(p._filename==fn) & (p._page==pg)]

In [ ]:
# predictions has one row per file. relation_name and relation_address may be a list
full_fn = p._filename+"_"+p._page.astype(str)
print(p.shape[0])
print(full_fn.nunique())

In [ ]:
# main performance comparison
files = gt_csv.Document.unique()
p_count = []
gt_count = []
fp_all = []
fn_all = []
matched_all = []
v_flag = False # verbose flag
for file in files:
    # get ground truth from manual annotation
    ground_truth = gt_csv[gt_csv.Document==file]

    # find predictions for the same page
    if "_" in file:
        fn, pg = os.path.splitext(file)[0].split("_")
        fn = fn[2:]
        predictions = p[(p._filename.str.contains(fn)) & (p._page==int(pg))]
    else:
        fn = os.path.splitext(file)[0]
        fn = fn[2:]
        predictions = p[(p._filename.str.contains(fn))]
    p_len = predictions.shape[0]

    # calculate performance
    matched_df, fp_df, fn_df = evaluate_predictions(predictions, ground_truth)

    if v_flag:
        print("GROUND TRUTH:")
        print(ground_truth.to_markdown())
    if p_len>0:
        if v_flag:
            print("PREDICTION:")  
            print(predictions.to_markdown())
       
    else:
        print("Not found in predictions")
    gt_count.append(ground_truth.shape[0])
    p_count.append(p_len)
    fp_all.append(fp_df)
    fn_all.append(fn_df)
    matched_all.append(matched_df)

    
    if v_flag:
        print("\n\n")
matched_df = pd.concat(matched_all).sort_values(by=["pred__filename", "pred__page"])
fp_df = pd.concat(fp_all).sort_values(by=["pred__filename", "pred__page"])
fn_df = pd.concat(fn_all).sort_values(by="gt_Document")

In [ ]:
print("Matched records: %d" % matched_df.shape[0])
print("Records in ground truth not found in predictions: % d" % fn_df.shape[0])
print("Records in predictions not found in GT: % d" % fp_df.shape[0])

In [ ]:
metrics = calc_metrics(fp_df, matched_df, fn_df)
print("── Metrics ──────────────────────────────")
for k, v in metrics.items():
    print(f"  {k}: {v}")


In [ ]:
print(fp_df.to_markdown())

In [ ]:
print(fn_df.sort_values(by="gt_Document").to_markdown())

In [ ]:
counts = pd.DataFrame({"pred":p_count, "gt": gt_count})
print("Correct number of entities in %2.1f percent of the pages" % (100*(counts["gt"]==counts.pred).sum()/counts.shape[0]))
print("Model found less entities than manual annotation: %2.1f percent of the pages" % (100*(counts["gt"]>counts.pred).sum()/counts.shape[0]))
print("Model found more entities than manual annotation: %2.1f percent of the pages" % (100*(counts["gt"]<counts.pred).sum()/counts.shape[0]))


In [ ]:
(counts["gt"]==counts.pred).sum()/counts.shape[0]

In [ ]:
file = gt_csv.Document.iloc[0]
fn = os.path.splitext(file)[0]
fn = fn[2:]
predictions = p[(p._filename.str.contains(fn))]

In [ ]:
predictions.relation_name

In [ ]:
p_count
gt_count


In [ ]:
# from predictions to ground truth json
idx = 1
gt_file = os.path.splitext(p.loc[idx]._filename)[0] + "_" + str(p.loc[idx]._page) + ".json"
p[(p._filename==fn) & (p._page==pg)]

In [ ]:
# using json
files = os.listdir(ground_truth_folder)
    
for file in files[:5]:
    print("Ground truth file: %s" % file)
    if "_" in file:
        fn, pg = os.path.splitext(file)[0].split("_")
        predictions = p[(p._filename==fn+".pdf") & (p._page==int(pg))]
    else:
        fn = os.path.splitext(file)[0]
        predictions = p[(p._filename==fn+".pdf")]
    if predictions.shape[0]>0:
        print("Found %d rows in predictions: index - %d" % (predictions.shape[0], int(predictions.index[0])))
        print("Company names in predictions: %s" % ", ".join(predictions.company_name.unique().tolist()))
    else:
        print("Not found in predictions")
    gt_file_path = ground_truth_folder / file
    with open(gt_file_path, 'r') as f:
        ground_truth = json.load(f)
    print("Ground truth company name: %s" % ground_truth["entities"]["company"]["name"])
    print("Ground truth number of entities: %d" % len(ground_truth["entities"]["role_holders"]))

    print(predictions.to_markdown())
    print(ground_truth["entities"])
    print("\n\n")
#predictions = 
#calculate_performance(predictions, ground_truth)



In [ ]:
! pip install tabulate

In [ ]:
p[p._filename.str.contains("<document-id>")]._filename.tolist()
